In [0]:
import pandas as pd


In [0]:
!pip install mne
import mne

In [0]:
!pip install mne-qt-browser PyQt5 -q
import mne_qt_browser
mne.viz.set_browser_backend('qt')

In [0]:
import numpy as np
import base64
import mne
import os

# Reload raw data if needed
raw = mne.io.read_raw_edf('/Volumes/kumc_sleep/sleep_studies/shhs_data/shhs1-204773.edf')

# =============================================
# USER INPUT: lights_off (seconds into recording)
# This defines t=0 on the displayed time axis.
# All times will be shown relative to this point.
# =============================================
lights_off = 330  # <-- Set this to the lights-off time in seconds

# Extract the EDF filename from the raw object
edf_filepath = raw.filenames[0]
edf_filename = os.path.basename(edf_filepath)

print(f"EDF file: {edf_filename}")
print(f"Full path: {edf_filepath}")
print(f"Lights off: {lights_off}s (t=0 reference)")

# Select channels
picks = ["ECG", "EOG(L)", "EMG", "EEG", "SaO2", "THOR RES", "ABDO RES"]
available_picks = [ch for ch in picks if ch in raw.ch_names]
n_channels = len(available_picks)

# Parameters - full 125 Hz resolution
sfreq = raw.info['sfreq']  # 125 Hz
target_sfreq = 125  # NO downsampling - full resolution
epoch_duration = 30  # seconds
samples_per_epoch = int(epoch_duration * target_sfreq)  # 3750
total_duration = raw.n_times / sfreq
n_epochs = int(np.floor(total_duration / epoch_duration))

print(f"Channels: {available_picks}")
print(f"Sample rate: {target_sfreq} Hz (full resolution)")
print(f"Total epochs: {n_epochs}, Samples per epoch: {samples_per_epoch}")

# Load all data at full resolution (no downsampling)
data_arr, times = raw[available_picks, :int(n_epochs * epoch_duration * sfreq)]

print(f"Data shape: {data_arr.shape}")

# Encode data as base64 Float32
channel_data_b64 = []
for i in range(n_channels):
    ch_data = data_arr[i].astype(np.float32)
    b64 = base64.b64encode(ch_data.tobytes()).decode('ascii')
    channel_data_b64.append(b64)

print(f"Encoding complete. Estimated file size: {sum(len(b) for b in channel_data_b64) / (1024*1024):.1f} MB (base64 data)")

# Build HTML with proper subplot separation
html_content = f"""<!DOCTYPE html>
<html>
<head>
<title>EDF Epoch Viewer - {edf_filename} (125 Hz)</title>
<script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
<style>
  body {{ margin: 0; padding: 10px; background: #1a1a2e; color: #eee; font-family: Arial, sans-serif; }}
  #controls {{ text-align: center; padding: 10px; background: #16213e; border-radius: 8px; margin-bottom: 10px; }}
  #file-info {{ font-size: 18px; font-weight: bold; margin-bottom: 6px; color: #00d4ff; }}
  #controls button {{ margin: 0 5px; padding: 8px 16px; background: #0f3460; color: #eee; border: 1px solid #444; border-radius: 4px; cursor: pointer; font-size: 14px; }}
  #controls button:hover {{ background: #1a5276; }}
  #epoch-info {{ font-size: 16px; margin: 8px 0; }}
  #slider-container {{ margin: 8px 0; }}
  #epoch-slider {{ width: 80%; }}
  #plot {{ width: 100%; height: calc(100vh - 150px); }}
</style>
</head>
<body>
<div id="controls">
  <div id="file-info">{edf_filename}</div>
  <div id="epoch-info">Epoch 1 / {n_epochs} | Time: 0s - 30s | 125 Hz</div>
  <button onclick="go(-10)">&#x25C0;10</button>
  <button onclick="go(-1)">&#x25C0; Prev</button>
  <button onclick="go(1)">Next &#x25B6;</button>
  <button onclick="go(10)">10&#x25B6;</button>
  <div id="slider-container">
    <input type="range" id="epoch-slider" min="0" max="{n_epochs-1}" value="0" oninput="jumpTo(this.value)">
  </div>
</div>
<div id="plot"></div>
<script>
const nCh = {n_channels};
const nEpochs = {n_epochs};
const samplesPerEpoch = {samples_per_epoch};
const targetSfreq = {target_sfreq};
const chNames = {available_picks};
const lightsOff = {lights_off};  // t=0 reference point in recording seconds

// Decode base64 channel data
const channelDataB64 = [
"""

for i, b64 in enumerate(channel_data_b64):
    html_content += f'  "{b64}"'
    if i < n_channels - 1:
        html_content += ','
    html_content += '\n'

html_content += f"""];

const channelData = channelDataB64.map(b64 => {{
  const binary = atob(b64);
  const bytes = new Uint8Array(binary.length);
  for (let i = 0; i < binary.length; i++) bytes[i] = binary.charCodeAt(i);
  return new Float32Array(bytes.buffer);
}});

let currentEpoch = 0;

function renderEpoch(epoch) {{
  const startSample = epoch * samplesPerEpoch;
  const startTime = epoch * 30;  // absolute recording seconds
  const endTime = startTime + 30;
  
  // Displayed time is relative to lights_off (t=0)
  const displayStart = startTime - lightsOff;
  const displayEnd = endTime - lightsOff;
  
  // Generate time axis using displayed time (relative to lights_off)
  const timeAxis = [];
  for (let i = 0; i < samplesPerEpoch; i++) {{
    timeAxis.push(displayStart + i / targetSfreq);
  }}
  
  // Create tick values every 5 seconds, labeled in plain seconds
  const tickVals = [];
  const tickTexts = [];
  for (let t = 0; t <= 30; t += 5) {{
    const displayVal = displayStart + t;
    tickVals.push(displayVal);
    tickTexts.push(displayVal + 's');
  }}
  
  const traces = [];
  const layout = {{
    paper_bgcolor: '#1a1a2e',
    plot_bgcolor: '#0f3460',
    font: {{ color: '#eee' }},
    margin: {{ l: 80, r: 30, t: 10, b: 50 }},
    showlegend: false,
  }};
  
  // Calculate domains for each channel - equal height with gaps
  const gap = 0.02;  // 2% gap between subplots
  const totalGap = gap * (nCh - 1);
  const plotHeight = (1.0 - totalGap) / nCh;
  
  for (let i = 0; i < nCh; i++) {{
    const chData = channelData[i].slice(startSample, startSample + samplesPerEpoch);
    
    // Each channel gets its own xaxis and yaxis
    const xAxisRef = i === 0 ? 'x' : 'x' + (i + 1);
    const yAxisRef = i === 0 ? 'y' : 'y' + (i + 1);
    
    traces.push({{
      x: timeAxis,
      y: Array.from(chData),
      type: 'scattergl',
      mode: 'lines',
      line: {{ width: 1, color: ['#00d4ff','#ff6b6b','#ffd93d','#6bcf7f','#c56cf0','#ff9f43','#54a0ff'][i] }},
      xaxis: xAxisRef,
      yaxis: yAxisRef,
      name: chNames[i]
    }});
    
    // Domain: channel 0 at top, channel nCh-1 at bottom
    const domainBottom = 1.0 - (i + 1) * plotHeight - i * gap;
    const domainTop = 1.0 - i * plotHeight - i * gap;
    
    layout['yaxis' + (i === 0 ? '' : (i + 1))] = {{
      domain: [domainBottom, domainTop],
      title: {{ text: chNames[i], font: {{ size: 11 }} }},
      gridcolor: '#1a3a5c',
      zerolinecolor: '#2a4a6c',
      autorange: true,
      fixedrange: true
    }};
    
    layout['xaxis' + (i === 0 ? '' : (i + 1))] = {{
      anchor: yAxisRef,
      range: [displayStart, displayEnd],
      showticklabels: i === nCh - 1,  // only show on bottom
      tickvals: i === nCh - 1 ? tickVals : undefined,
      ticktext: i === nCh - 1 ? tickTexts : undefined,
      gridcolor: '#1a3a5c',
      title: i === nCh - 1 ? {{ text: 'Time (seconds, t=0 at lights off) \u2014 Epoch ' + (epoch + 1) + ' / ' + nEpochs, font: {{ size: 12 }} }} : undefined,
      fixedrange: true,
      matches: i === 0 ? undefined : 'x'  // sync all x axes
    }};
  }}
  
  Plotly.react('plot', traces, layout, {{ responsive: true, displayModeBar: false }});
  
  document.getElementById('epoch-info').textContent = 
    'Epoch ' + (epoch + 1) + ' / ' + nEpochs + ' | Time: ' + displayStart + 's to ' + displayEnd + 's | 125 Hz';
  document.getElementById('epoch-slider').value = epoch;
}}

function go(delta) {{
  currentEpoch = Math.max(0, Math.min(nEpochs - 1, currentEpoch + delta));
  renderEpoch(currentEpoch);
}}

function jumpTo(val) {{
  currentEpoch = parseInt(val);
  renderEpoch(currentEpoch);
}}

document.addEventListener('keydown', function(e) {{
  if (e.key === 'ArrowRight' || e.key === 'n') go(1);
  else if (e.key === 'ArrowLeft' || e.key === 'p') go(-1);
  else if (e.key === 'Home') {{ currentEpoch = 0; renderEpoch(0); }}
  else if (e.key === 'End') {{ currentEpoch = nEpochs - 1; renderEpoch(currentEpoch); }}
}});

// Initial render
renderEpoch(0);
</script>
</body>
</html>"""

# Write to file
output_path = "/Workspace/Users/gpuchalski@kumc.edu/edf_epoch_viewer.html"
with open(output_path, 'w') as f:
    f.write(html_content)

file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"\nEpoch viewer saved to: {output_path}")
print(f"File size: {file_size_mb:.1f} MB")
print(f"Viewing: {edf_filename}")
print(f"Lights off (t=0): {lights_off}s")
print(f"X-axis: seconds relative to lights_off (negative = before lights off)")

In [0]:
data = pd.read_csv("/Volumes/kumc_sleep/sleep_studies/shhs_data/excel_sheet/sleep_excel_ordered.csv")
print(data.columns)

------------------------------
## Sleep Heart Health Study (SHHS) Variable Dictionary
These variables are primarily used to study the relationship between Obstructive Sleep Apnea (OSA) and Cardiovascular Disease (CVD).
## 1. Identification & Study Timeline

| Variable | Description |
|---|---|
| nsrrid | National Sleep Research Resource ID: Unique subject identifier. |
| fudays / fuyears | Follow-up Time: Total duration (days or years) the subject was tracked. |
| censdate | Censoring Date: The date data collection ended for that specific subject. |
| vital | Vital Status: Indicates if the subject was alive or deceased at the end of follow-up. |

## 2. Baseline Health (Pre-existing Conditions)
Variables ending in _01 are typically binary (1 = Yes, 0 = No).

| Variable | Description |
|---|---|
| prev_cvd_all_01 | Any Pre-existing Cardiovascular Disease |
| prev_chd_all_01 | Pre-existing Coronary Heart Disease |
| prev_chf_01 | Pre-existing Congestive Heart Failure |
| prev_stk_01 | Pre-existing History of Stroke |
| dm01_s1 | Diabetes Mellitus at baseline |
| htn01_s1 | Hypertension at baseline |

## 3. Incident Events (New Health Outcomes)
These track events that occurred after the initial sleep study.

| Variable | Description |
|---|---|
| inci_cvd_all_01 | Incident (new) Cardiovascular Disease |
| inci_cvd_death_01 | Death attributed to Cardiovascular Disease |
| mace_allcausedeath | Major Adverse Cardiovascular Event or death from any cause |
|  _time suffixes (e.g., inci_stk_time) | The number of days until that specific event occurred |

## 4. Sleep & Respiratory Metrics

| Variable | Description |
|---|---|
| slptime / slptime_s1 | Sleep Time: Total minutes/hours of sleep during the study. |
| ahi_a0h4_s1 | Apnea-Hypopnea Index: Number of breathing pauses per hour (4% oxygen desaturation threshold). |
| OSA_category_s1 | OSA Severity: Categorized (e.g., None, Mild, Moderate, Severe). |
| MinSat | Minimum Oxygen Saturation: The lowest O2 level recorded during sleep. |
| pctlt90 | Percent Below 90%: Percentage of sleep time spent with O2 saturation < 90%. |
| total_ap_hyp_count | Event Count: Total number of apneas and hypopneas. |

## 5. Hypoxic Burden & Exposure
These measure the "dose" of low oxygen the body receives during sleep.

| Variable | Description |
|---|---|
| HypoxicBurden (HB) | The cumulative area under the desaturation curve; measures the total "load" of oxygen drops |
| HypoxicExposure | A similar metric quantifying the duration and depth of oxygen desaturations |
| HB4_Philip | A specific calculation of hypoxic burden (likely derived from the Philip et al. method) |
| _log / _log10 / _cat | These indicate the data was transformed (Logarithmic scale) or categorized for statistical modeling |

## 6. Demographics & Biomarkers

| Variable | Description |
|---|---|
| age_s1 / gender / bmi_s1 | Age, Gender, and Body Mass Index at the time of the first study |
| race_s1 / ethnicity_s1 | Self-reported race and ethnicity |
| smokecat_s1 / alcoh | Smoking status (category) and alcohol consumption |
| hdl_s1 / chol_s1 / trig_s1 | Blood lipid levels (HDL, Total Cholesterol, Triglycerides) |
| ess_s1 / essgt10 | Epworth Sleepiness Scale score; essgt10 indicates if the score is greater than 10 (excessive daytime sleepiness) |





In [0]:
cols = [
    "prev_cvd_all_01", "prev_chd_all_01", "prev_chf_01", "prev_stk_01",
    "inci_cvd_death_01", "inci_cvd_all_01", "inci_chd_all_01", "inci_chf_01", "inci_stk_01",
    "new_cvd_all_01", "new_chd_all_01", "new_chf_01", "new_stk_01",
    "dm01_s1", "htn01_s1", "lipid1_s1", "Diabetes", "COPD", "ES_vs_others"
]

summary = []
for col in cols:
    counts = data[col].value_counts(dropna=False)
    yes = counts.get("Yes", 0)
    no = counts.get("No", 0)
    nans = data[col].isna().sum()
    total = yes + no + nans
    yes_pct = (yes / total * 100) if total > 0 else 0
    no_pct = (no / total * 100) if total > 0 else 0
    nan_pct = (nans / total * 100) if total > 0 else 0
    summary.append({
        "column": col,
        "Yes %": yes_pct,
        "No %": no_pct,
        "Nan %": nan_pct,
        "Yes_count": yes,
        "No_count": no,
        "NaN_count": nans
    })

summary_df = pd.DataFrame(summary)
display(summary_df)

gender_counts = data["gender"].value_counts(dropna=False)
male = gender_counts.get("Male", 0)
female = gender_counts.get("Female", 0)
total_gender = male + female
male_pct = (male / total_gender * 100) if total_gender > 0 else 0
female_pct = (female / total_gender * 100) if total_gender > 0 else 0

gender_summary = pd.DataFrame({
    "Gender": ["Male", "Female"],
    "Count": [male, female],
    "%": [male_pct, female_pct]
})
display(gender_summary)

race_counts = data["race_s1"].value_counts(dropna=False)
white = race_counts.get("White", 0)
black = race_counts.get("Black", 0)
other = race_counts.get("Other", 0)
total_race = white + black + other

white_pct = (white / total_race * 100) if total_race > 0 else 0
black_pct = (black / total_race * 100) if total_race > 0 else 0
other_pct = (other / total_race * 100) if total_race > 0 else 0

# For "Other", calculate percentage of "Hispanic or Latino" in ethnicity_s1
other_indices = data["race_s1"] == "Other"
other_ethnicity = data.loc[other_indices, "ethnicity_s1"]
hispanic_count = (other_ethnicity == "Hispanic or Latino").sum()
other_total = other_indices.sum()
hispanic_pct = (hispanic_count / other_total * 100) if other_total > 0 else 0

race_summary = pd.DataFrame({
    "Race": ["White", "Black", "Other"],
    "Count": [white, black, other],
    "%": [white_pct, black_pct, other_pct],
    "Hispanic/Latino % in Other": [None, None, hispanic_pct]
})
display(race_summary)

smoke_counts = data["smokecat_s1"].value_counts(dropna=False)
former = smoke_counts.get("Former", 0)
never = smoke_counts.get("Never", 0)
current = smoke_counts.get("Current", 0)
total_smoke = former + never + current

former_pct = (former / total_smoke * 100) if total_smoke > 0 else 0
never_pct = (never / total_smoke * 100) if total_smoke > 0 else 0
current_pct = (current / total_smoke * 100) if total_smoke > 0 else 0

smoke_summary = pd.DataFrame({
    "Smoking Status": ["Former", "Never", "Current"],
    "Count": [former, never, current],
    "%": [former_pct, never_pct, current_pct]
})
display(smoke_summary)

for col in ["LC_4", "LC4_OSA_combination", "LC4_OSA_combination_excl_Mild"]:
    minimally = data[col].str.contains("Minimally", na=False).sum()
    excessively = data[col].str.contains("Excessively", na=False).sum()
    disturbed = data[col].str.contains("Disturbed", na=False).sum()
    moderately = data[col].str.contains("Moderately", na=False).sum()
    na_count = data[col].isna().sum()
    total = minimally + excessively + disturbed + moderately

    minimally_pct = (minimally / total * 100) if total > 0 else 0
    excessively_pct = (excessively / total * 100) if total > 0 else 0
    disturbed_pct = (disturbed / total * 100) if total > 0 else 0
    moderately_pct = (moderately / total * 100) if total > 0 else 0
    na_pct = (na_count / (total + na_count) * 100) if (total + na_count) > 0 else 0

    summary_df = pd.DataFrame({
        "Category": ["Minimally", "Excessively", "Disturbed", "Moderately", "NA"],
        "Count": [minimally, excessively, disturbed, moderately, na_count],
        "%": [minimally_pct, excessively_pct, disturbed_pct, moderately_pct, na_pct]
    })
    summary_df.insert(0, "Column", col)
    display(summary_df)

osa_cols = ["IsNoOSA", "IsModSev", "IsMild", "IsES", "IsDS", "IsModS", "IsMS"]

osa_summary = []
for col in osa_cols:
    counts = data[col].value_counts(dropna=False)
    true_count = counts.get(True, 0)
    false_count = counts.get(False, 0)
    na_count = data[col].isna().sum()
    total = true_count + false_count
    true_pct = (true_count / total * 100) if total > 0 else 0
    false_pct = (false_count / total * 100) if total > 0 else 0
    na_pct = (na_count / (total + na_count) * 100) if (total + na_count) > 0 else 0
    osa_summary.append({
        "column": col,
        "True %": true_pct,
        "False %": false_pct,
        "NaN %": na_pct,
        "True_count": true_count,
        "False_count": false_count,
        "NaN_count": na_count,
    })

osa_summary_df = pd.DataFrame(osa_summary)
display(osa_summary_df)

for col in ["ES_OSA_combination", "ES_OSA_combination_excl_Mild"]:
    not_count = data[col].str.contains("Not", na=False).sum()
    excessively_count = data[col].apply(lambda x: not (pd.isna(x) or ("Not" in str(x)))).sum()
    na_count = data[col].isna().sum()
    total = not_count + excessively_count
    not_pct = (not_count / (total + na_count) * 100) if (total + na_count) > 0 else 0
    excessively_pct = (excessively_count / (total + na_count) * 100) if (total + na_count) > 0 else 0
    na_pct = (na_count / (total + na_count) * 100) if (total + na_count) > 0 else 0
    summary_df = pd.DataFrame({
        "Category": ["Not", "Excessively", "NA"],
        "Count": [not_count, excessively_count, na_count],
        "%": [not_pct, excessively_pct, na_pct]
    })
    display(summary_df)

col = "ESSgt10_OSA_combination"
counts = data[col].value_counts(dropna=False)
modsev_ess_le10 = counts.get("Moderate-severe OSA ESS<=10", 0)
modsev_ess_gt10 = counts.get("Moderate-severe OSA ESS>10", 0)
na_count = data[col].isna().sum()
total = modsev_ess_le10 + modsev_ess_gt10
total_with_na = total + na_count

modsev_ess_le10_pct = (modsev_ess_le10 / total_with_na * 100) if total_with_na > 0 else 0
modsev_ess_gt10_pct = (modsev_ess_gt10 / total_with_na * 100) if total_with_na > 0 else 0
na_pct = (na_count / total_with_na * 100) if total_with_na > 0 else 0

essgt10_summary = pd.DataFrame({
    "Category": ["Moderate-severe OSA ESS<=10", "Moderate-severe OSA ESS>10", "NA"],
    "Count": [modsev_ess_le10, modsev_ess_gt10, na_count],
    "%": [modsev_ess_le10_pct, modsev_ess_gt10_pct, na_pct]
})
display(essgt10_summary)

hypoxic_cols = [
    "HypoxicBurden.cat", "HypoxicExposure.cat", "HypoxicExposure_v2.cat",
    "HypoxicExposure_v3.cat", "HB4_Philip.cat"
]

for col in hypoxic_cols:
    counts = data[col].value_counts(dropna=False)
    q5 = counts.get("Q5", 0)
    q4 = counts.get("Q4", 0)
    q3 = counts.get("Q3", 0)
    q2 = counts.get("Q2", 0)
    q1 = counts.get("Q1", 0)
    na_count = data[col].isna().sum()
    total = q5 + q4 + q3 + q2 + q1
    total_with_na = total + na_count

    q5_pct = (q5 / total_with_na * 100) if total_with_na > 0 else 0
    q4_pct = (q4 / total_with_na * 100) if total_with_na > 0 else 0
    q3_pct = (q3 / total_with_na * 100) if total_with_na > 0 else 0
    q2_pct = (q2 / total_with_na * 100) if total_with_na > 0 else 0
    q1_pct = (q1 / total_with_na * 100) if total_with_na > 0 else 0
    na_pct = (na_count / total_with_na * 100) if total_with_na > 0 else 0

    summary_df = pd.DataFrame({
        "Category": ["Q5", "Q4", "Q3", "Q2", "Q1", "NA"],
        "Count": [q5, q4, q3, q2, q1, na_count],
        "%": [q5_pct, q4_pct, q3_pct, q2_pct, q1_pct, na_pct]
    })
    summary_df.insert(0, "Column", col)
    display(summary_df)


from scipy import stats
import numpy as np

cols_stats = [
    "inci_cvd_death_time", "inci_cvd_all_time", "inci_chd_all_time", "inci_chf_time", "inci_stk_time",
    "new_cvd_all_time", "new_chd_all_time", "new_chf_time", "new_stk_time", "fudays", "fuyears",
    "age_s1", "bmi_s1", "hdl_s1", "chol_s1", "trig_s1", "slptime_s1", "ahi_a0h4_s1", "OSA_category_s1",
    "ess_s1", "slptime", "total_ap_hyp_count", "HypoxicBurden", "TST.sec", "HypoxicExposure",
    "HypoxicExposure_v2", "HypoxicExposure_v3", "alcoh", "ahi_a0h3a", "pctlt90", "MinSat", "NDes3pH",
    "HypoxicBurden_logn", "HypoxicBurden.log", "HypoxicExposure.log", "HypoxicExposure_v2.log",
    "HypoxicExposure_v3.log", "HB4_Philip", "vital", "censdate", "mace_allcausedeath", "mace_acm_time",
    "HypoxicExposure_v2.log10", "HypoxicBurden.log10", "HB4_Philip.log", "HB4_Philip.log10"
]


stats_summary = []
for col in cols_stats:
    if col not in data.columns:
        continue
    x = pd.to_numeric(data[col], errors='coerce')
    n = x.count()
    nans = x.isna().sum()
    pct_nans = nans / len(x) * 100 if len(x) > 0 else np.nan
    mean = x.mean()
    median = x.median()
    std = x.std()
    var = x.var()
    min_ = x.min()
    max_ = x.max()
    rng = max_ - min_
    iqr = stats.iqr(x, nan_policy='omit')
    mad = stats.median_abs_deviation(x, nan_policy='omit')
    skew = x.skew()
    kurt = x.kurt()
    coef_var = std / mean if mean not in [0, np.nan] else np.nan
    mode = x.mode(dropna=True)
    mode_val = mode.iloc[0] if not mode.empty else np.nan
    zscore = ((x - mean) / std).median() if std not in [0, np.nan] else np.nan
    med = median
    mad_mod = mad * 1.4826
    mod_zscore = (0.6745 * (x - med) / mad_mod).median() if mad_mod not in [0, np.nan] else np.nan
    q1 = x.quantile(0.25)
    q3 = x.quantile(0.75)
    iqr_fence_low = q1 - 1.5 * iqr
    iqr_fence_high = q3 + 1.5 * iqr

    stats_summary.append({
        "column": col,
        "mean": mean,
        "median": median,
        "mode": mode_val,
        "min": min_,
        "max": max_,
        "range": rng,
        "std": std,
        "variance": var,
        "iqr": iqr,
        "mad": mad,
        "skewness": skew,
        "kurtosis": kurt,
        "coefficient_of_variance": coef_var,
        "percent_nans": pct_nans,
        "zscore_median": zscore,
        "modified_zscore_median": mod_zscore,
        "iqr_fence_low": iqr_fence_low,
        "iqr_fence_high": iqr_fence_high
    })

stats_df = pd.DataFrame(stats_summary)
display(stats_df)

import matplotlib.pyplot as plt

hist_cols = [
    "inci_cvd_death_time", "inci_cvd_all_time", "inci_chd_all_time", "inci_chf_time", "inci_stk_time",
    "new_cvd_all_time", "new_chd_all_time", "new_chf_time", "new_stk_time", "fudays", "fuyears",
    "age_s1", "bmi_s1", "hdl_s1", "chol_s1", "trig_s1", "slptime_s1", "ahi_a0h4_s1", "OSA_category_s1",
    "ess_s1", "slptime", "total_ap_hyp_count", "HypoxicBurden", "TST.sec", "HypoxicExposure",
    "HypoxicExposure_v2", "HypoxicExposure_v3", "alcoh", "ahi_a0h3a", "pctlt90", "MinSat", "NDes3pH",
    "HypoxicBurden_logn", "HypoxicBurden.log", "HypoxicExposure.log", "HypoxicExposure_v2.log",
    "HypoxicExposure_v3.log", "HB4_Philip", "vital", "censdate", "mace_allcausedeath", "mace_acm_time",
    "HypoxicExposure_v2.log10", "HypoxicBurden.log10", "HB4_Philip.log", "HB4_Philip.log10"
]

n_cols = 4
n_rows = int(np.ceil(len(hist_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(hist_cols):
    if col in data.columns:
        data[col].dropna().hist(ax=axes[i], bins=30)
        axes[i].set_title(col)
    else:
        axes[i].axis('off')

plt.tight_layout()
plt.show()

import seaborn as sns

corr_matrix = data[hist_cols].corr()
plt.figure(figsize=(16, 12))
sns.heatmap(corr_matrix, cmap="coolwarm", annot=True, fmt=".2f", linewidths=0.5)
plt.title("Correlation Heatmap for hist_cols")
plt.tight_layout()
plt.show()

In [0]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.scatter(data["age_s1"], data["ahi_a0h4_s1"], alpha=0.6)
plt.ylabel("ahi_a0h4_s1")
plt.xlabel("age_s1")
plt.title("age_s1 vs ahi_a0h4_s1")

plt.subplot(1, 2, 2)
plt.scatter(data["bmi_s1"], data["ahi_a0h4_s1"], alpha=0.6)
plt.ylabel("ahi_a0h4_s1")
plt.xlabel("bmi_s1")
plt.title("bmi_s1 vs ahi_a0h4_s1")

plt.tight_layout()
plt.show()

# Statistical Metrics: Interpretation Guide for Numeric Data
 
These metrics describe different structural properties of a dataset’s distribution — its spread, symmetry, tail behavior, robustness to outliers, and multiplicative structure. Together they provide a comprehensive understanding beyond simple averages.
 
---
 
# Interquartile Range (IQR) 
**Definition**
 
IQR = Q3 − Q1
 
where:
- Q1 = 25th percentile
- Q3 = 75th percentile 

**Measures:** 
Spread of the middle 50% of the data.

**What it tells you:**
Resistant to outliers, Useful for skewed datasets, Preferred over standard deviation when distributions are non-normal
 
**Interpretation**
 
| IQR size | Meaning |
|----------|---------|
| Small | Values tightly clustered |
| Large | High variability in central observations |
 
---
 
# Median Absolute Deviation (MAD)
 
**Definition**
 
MAD = median( |xi − median(x)| )
 
**Measures:**
Robust spread around the median.
 
**What it tells you:** Resistant to extreme values, Preferred for heavy-tailed distributions, Useful in anomaly detection
 
Think of MAD as a robust alternative to standard deviation.
 
---
 
# Variance
 
**Definition**
 
Var(X) = (1 / (n − 1)) Σ(xi − mean)²
 
**Measures:** Average squared deviation from the mean.
 
**What it tells you: **Overall variability of the dataset,Sensitive to outliers,Foundation of regression, PCA, and hypothesis testing
 
Large variance indicates heterogeneous data.
 
---
 
# Skewness
 
**Definition**
 
Measures asymmetry of a distribution.
 
**Interpretation**
 
| Skewness | Meaning |
|----------|---------|
| ≈ 0 | Symmetric distribution |
| > 0 | Right tail longer |
| < 0 | Left tail longer |
 
Example:

- Income distributions → positive skew

- Survival time data → often positive skew
 
---
 
# Kurtosis
 
**Definition**
 
Measures tail heaviness (frequency of extreme values).
 
**Interpretation**
 
| Kurtosis | Meaning |
|----------|---------|
| ≈ 0 | Normal distribution tail weight |
| > 0 | Heavy tails (more extreme values) |
| < 0 | Light tails |
 
High kurtosis indicates increased outlier risk.
 
---
 

 
# Coefficient of Variation (CV)
 
**Definition**
 
CV = standard deviation / mean
 
**Measures:** Relative variability independent of scale.
 
**What it tells you:** Allows comparison across variables with different units.
 
Example:
 
| Variable | Mean | Std Dev | CV |
|----------|------|---------|----|
| Blood pressure | 120 | 10 | Low variability |
| Triglycerides | 120 | 60 | High variability |
 
---
 
# Z-score (Mean-Based)
 
**Definition**
 
z = (x − mean) / standard deviation
 
**Measures:** Distance from the mean in standard deviation units.
 
**Interpretation**
 
|Z-Score | Meaning |
|--|---------|
|`abs(z) < 2` | Typical |
| `2 ≤ abs(z) < 3` | Unusual |
| `abs(z) ≥ 3` | Likely outlier |
 
Best used for approximately normal distributions.
 
---
 
# Modified Z-score (Median-Based)
 
**Definition**
 
Modified z = 0.6745 × (x − median) / MAD
 
**Measures:** Robust outlier detection.
 
**Why it is useful:** Uses median instead of mean, making it resistant to extreme values.
 
**Interpretation**
 
| Value | Meaning |
|-------|---------|
| > 3.5 | Strong outlier candidate |
 
Common in biomedical datasets.
 
---
 
# IQR Fences (Outlier Thresholds)
 
**Definitions**
 
Lower fence = Q1 − 1.5 × IQR  

Upper fence = Q3 + 1.5 × IQR
 
**Measures:** Thresholds identifying percentile-based outliers.
 
**Interpretation**
 
| Region | Meaning |
|--------|---------|
| Below lower fence | Lower outliers |
| Above upper fence | Upper outliers |
 
Used in boxplots and robust screening pipelines.
 
---
 
# How These Metrics Work Together
 
| Metric | Detects |
|--------|---------|
| Variance | Overall spread |
| IQR | Robust central spread |
| MAD | Robust variability |
| Skewness | Asymmetry |
| Kurtosis | Tail risk |
| Geometric mean | Multiplicative structure |
| Harmonic mean | Rate averaging |
| CV | Relative variability |
| Z-score | Gaussian outliers |
| Modified Z-score | Robust outliers |
| IQR fences | Percentile-based outliers |
 
Together they describe location, scale, symmetry, tail behavior, and anomaly structure of a dataset.
 
---
 
# References
 
- https://www.itl.nist.gov/div898/handbook/eda/section3/eda35b.htm

- https://online.stat.psu.edu/stat200/lesson/3/3.2

- https://docs.scipy.org/doc/scipy/reference/stats.html

- https://www.stat.cmu.edu/~larry/=stat705/
1.3.5.11. Measures of Skewness and Kurtosis
 